# Programming Assignment: Missingness Pattern Analysis

Welcome to the **Missingness Pattern Analysis** assignment!

Counting missing values tells you *how much* data is missing; understanding **why** data is missing allows you to choose valid handling strategies. As the theory explains, Donald Rubin (1976) classifies missingness into three mechanisms:

- **MCAR** — missingness is completely random, independent of all variables
- **MAR** — missingness depends on *observed* variables but not on the missing value itself
- **MNAR** — missingness depends on the missing value itself (most dangerous, hardest to detect)

**What You Will Do in This Assignment:**

* Create binary missingness indicator variables for each column
* Implement a simplified MCAR test using statistical comparisons across missingness groups
* Test for MAR patterns by examining relationships between indicators and observed values
* Compute pairwise missingness correlations between columns
* Build a heuristic classifier that identifies the likely missingness type for a given column

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

---

## Table of Contents
- [Imports](#imports)
- [1 - Understanding Missingness Patterns](#1---understanding-missingness-patterns)
    - [1.1 - Generating Sample Data with Known Mechanisms](#11---generating-sample-data)
    - **[Exercise 1 - `create_missingness_indicators`](#exercise-1---create_missingness_indicators)**
- [2 - Testing for MCAR](#2---testing-for-mcar)
    - [2.1 - The Logic Behind MCAR Testing](#21---the-logic-behind-mcar-testing)
    - **[Exercise 2 - `simplified_mcar_test`](#exercise-2---simplified_mcar_test)**
- [3 - Testing for MAR](#3---testing-for-mar)
    - [3.1 - Understanding MAR Detection](#31---understanding-mar-detection)
    - **[Exercise 3 - `test_mar_pattern`](#exercise-3---test_mar_pattern)**
- [4 - Missingness Correlations](#4---missingness-correlations)
    - [4.1 - Why Correlations Matter](#41---why-correlations-matter)
    - **[Exercise 4 - `compute_missingness_correlation`](#exercise-4---compute_missingness_correlation)**
- [5 - Classifying Missingness Type](#5---classifying-missingness-type)
    - **[Exercise 5 - `classify_missingness_type`](#exercise-5---classify_missingness_type)**

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

In [ ]:
import helper_utils
import unittests

<a name='1---understanding-missingness-patterns'></a>
## 1 - Understanding Missingness Patterns

<a name='11---generating-sample-data'></a>
### 1.1 - Generating Sample Data with Known Mechanisms

We will work with three datasets, each demonstrating one of the three missingness mechanisms:

1. **MCAR data** — `income` is randomly missing, independent of all variables (a coin-flip)
2. **MAR data** — `income` is more likely missing for *older* individuals (missingness driven by observed `age`)
3. **MNAR data** — `income` is more likely missing for *high earners* (missingness driven by the unobserved value itself)

The key insight from the theory: the only mechanism we can *directly test* is MCAR, because any deviation from a flat missingness probability will show up as group mean differences.

In [ ]:
# Generate datasets with each missingness mechanism
mcar_data = helper_utils.generate_mcar_data(n_samples=500, missing_rate=0.2)
mar_data  = helper_utils.generate_mar_data(n_samples=500, missing_rate=0.3)
mnar_data = helper_utils.generate_mnar_data(n_samples=500, missing_rate=0.3)

print("MCAR Data — missing income:", mcar_data['income'].isnull().sum(),
      f"({mcar_data['income'].isnull().mean()*100:.1f}%)")
print("MAR Data  — missing income:", mar_data['income'].isnull().sum(),
      f"({mar_data['income'].isnull().mean()*100:.1f}%)")
print("MNAR Data — missing income:", mnar_data['income'].isnull().sum(),
      f"({mnar_data['income'].isnull().mean()*100:.1f}%)")

In [ ]:
# MCAR: age distribution should look the same whether income is missing or not
print("MCAR Pattern — age distribution should be similar for both groups")
helper_utils.plot_missingness_vs_values(mcar_data, 'income', 'age')

In [ ]:
# MAR: higher age → more missing income; clear age shift between groups
print("MAR Pattern — older individuals should have more missing income")
helper_utils.plot_missingness_vs_values(mar_data, 'income', 'age')

<a name='exercise-1---create_missingness_indicators'></a>
### **Exercise 1 - `create_missingness_indicators`**

**Your Task:**

Implement `create_missingness_indicators` that creates a binary missingness indicator variable for every column in the input DataFrame.

**Requirements:**
- For each column in `df`, create a corresponding column `{col}_missing`
- Indicator is `1` if the value is missing (`NaN`), `0` otherwise
- Return a *new* DataFrame containing **only** the indicator columns

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** `df.isnull()` gives a boolean DataFrame where `True` = missing

**Step 2:** `.astype(int)` converts `True/False` → `1/0`

**Step 3:** Rename columns: `indicators.columns = [f"{c}_missing" for c in df.columns]`

</details>

In [ ]:
# GRADED FUNCTION: create_missingness_indicators
def create_missingness_indicators(df):
    """
    Create binary indicator variables for each column's missingness.

    Args:
        df (pd.DataFrame): Input DataFrame with potential missing values.

    Returns:
        pd.DataFrame: DataFrame with binary indicators (1=missing, 0=present).
                      Column names are '{original_name}_missing'.
    """
    ### START CODE HERE ###

    # Step 1: Boolean mask for missing values
    missing_mask = None

    # Step 2: Convert boolean to integer
    indicators = None

    # Step 3: Rename columns with '_missing' suffix
    indicators.columns = None

    ### END CODE HERE ###

    return indicators

In [ ]:
test_df = pd.DataFrame({
    'A': [1.0, 2.0, np.nan, 4.0],
    'B': [np.nan, 2.0, 3.0, np.nan],
    'C': [1.0, 2.0, 3.0, 4.0]
})
print("Original DataFrame:")
print(test_df)
print("\nMissingness Indicators:")
print(create_missingness_indicators(test_df))

In [ ]:
# Test your code!
unittests.exercise_1(create_missingness_indicators)

<a name='2---testing-for-mcar'></a>
## 2 - Testing for MCAR

<a name='21---the-logic-behind-mcar-testing'></a>
### 2.1 - The Logic Behind MCAR Testing

Little's MCAR test (from the theory) checks whether the **means of observed variables differ across subgroups defined by missingness patterns**. The formal hypothesis test is:

- $H_0$: Data is MCAR — means do not differ across missingness groups
- $H_1$: Data is **not** MCAR

**Decision rule:** If $p \leq 0.05$, reject $H_0$ and conclude the data is MAR or MNAR.

Here we implement a *simplified* version using a two-sample t-test: split the dataset into rows where a target column is missing vs. not missing, then test whether a predictor column's mean differs significantly between the two groups. A significant difference ($p < 0.05$) provides evidence **against** MCAR.

<a name='exercise-2---simplified_mcar_test'></a>
### **Exercise 2 - `simplified_mcar_test`**

**Your Task:**

Implement `simplified_mcar_test(df, target_col, predictor_col)` that tests whether the mean of `predictor_col` differs significantly between rows where `target_col` is missing vs. present.

**Requirements:**
- Use `scipy.stats.ttest_ind` for the two-sample t-test
- Return a tuple `(t_statistic, p_value)` — both as floats
- Low p-value ($< 0.05$) → evidence **against** MCAR

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Create a boolean mask for missing rows: `mask = df[target_col].isnull()`

**Step 2:** Split predictor into two groups:
```python
group_missing  = df.loc[mask,  predictor_col].dropna()
group_present  = df.loc[~mask, predictor_col].dropna()
```

**Step 3:** `stats.ttest_ind(group_missing, group_present)` returns a named tuple with `.statistic` and `.pvalue`

</details>

In [ ]:
# GRADED FUNCTION: simplified_mcar_test
def simplified_mcar_test(df, target_col, predictor_col):
    """
    Test whether the mean of predictor_col differs between rows where target_col is missing vs. present.

    Args:
        df (pd.DataFrame): Input DataFrame.
        target_col (str): Column whose missingness defines the two groups.
        predictor_col (str): Column whose mean is compared between groups.

    Returns:
        tuple: (t_statistic, p_value) — both floats.
    """
    ### START CODE HERE ###

    # Step 1: Boolean mask — True where target_col is missing
    mask = None

    # Step 2: Split predictor into two groups
    group_missing = None
    group_present = None

    # Step 3: Two-sample t-test
    t_stat, p_value = None, None

    ### END CODE HERE ###

    return (t_stat, p_value)

In [ ]:
# MCAR: t-test on age should show NO significant difference
t_mcar, p_mcar = simplified_mcar_test(mcar_data, 'income', 'age')
print(f"MCAR data — t={t_mcar:.3f}, p={p_mcar:.3f}")
print(f"  Interpretation: {'Evidence against MCAR' if p_mcar < 0.05 else 'Consistent with MCAR'}")

# MAR: t-test on age SHOULD show a significant difference
t_mar, p_mar = simplified_mcar_test(mar_data, 'income', 'age')
print(f"\nMAR data  — t={t_mar:.3f}, p={p_mar:.3f}")
print(f"  Interpretation: {'Evidence against MCAR (MAR/MNAR)' if p_mar < 0.05 else 'Consistent with MCAR'}")

In [ ]:
# Test your code!
unittests.exercise_2(simplified_mcar_test)

<a name='3---testing-for-mar'></a>
## 3 - Testing for MAR

<a name='31---understanding-mar-detection'></a>
### 3.1 - Understanding MAR Detection

From the theory — a practical complement to Little's test is **missingness indicator correlation**: create binary indicators and test their association with observed variables.

If a missingness indicator (e.g., `income_missing = 1` where income is absent) **correlates with a fully observed variable** like `age`, it means the probability of income being missing can be *predicted* from what you already know. That is the definition of MAR: missingness depends on observed data, not on the missing value itself.

We quantify this with a **point-biserial correlation** (Pearson correlation between a binary indicator and a continuous variable) or a t-test.

<a name='exercise-3---test_mar_pattern'></a>
### **Exercise 3 - `test_mar_pattern`**

**Your Task:**

Implement `test_mar_pattern(df, missing_col, predictor_col)` that tests the association between a missingness indicator and an observed predictor column.

**Requirements:**
- Create a binary indicator: `1` where `missing_col` is `NaN`, `0` otherwise
- Compute the **Pearson correlation** between the indicator and `predictor_col`
- Return a tuple `(correlation, p_value)`

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Create indicator: `indicator = df[missing_col].isnull().astype(int)`

**Step 2:** Drop rows where `predictor_col` itself is NaN (to compute valid correlation)

**Step 3:** `stats.pearsonr(indicator_aligned, predictor_aligned)` returns `(r, p_value)`

</details>

In [ ]:
# GRADED FUNCTION: test_mar_pattern
def test_mar_pattern(df, missing_col, predictor_col):
    """
    Test whether missingness in missing_col is correlated with observed values of predictor_col.

    Args:
        df (pd.DataFrame): Input DataFrame.
        missing_col (str): Column whose missingness indicator to create.
        predictor_col (str): Fully observed column to correlate against.

    Returns:
        tuple: (correlation, p_value) — both floats.
    """
    ### START CODE HERE ###

    # Step 1: Create binary missingness indicator
    indicator = None

    # Step 2: Align — drop rows where predictor_col is NaN
    valid_mask = None
    indicator_aligned = None
    predictor_aligned = None

    # Step 3: Compute Pearson correlation
    correlation, p_value = None, None

    ### END CODE HERE ###

    return (correlation, p_value)

In [ ]:
# MCAR: no correlation expected
r_mcar, p_mcar = test_mar_pattern(mcar_data, 'income', 'age')
print(f"MCAR — r={r_mcar:.3f}, p={p_mcar:.3f}: {'MAR signal detected' if p_mcar < 0.05 else 'No MAR signal'}")

# MAR: strong correlation expected
r_mar, p_mar = test_mar_pattern(mar_data, 'income', 'age')
print(f"MAR  — r={r_mar:.3f}, p={p_mar:.3f}: {'MAR signal detected' if p_mar < 0.05 else 'No MAR signal'}")

In [ ]:
# Test your code!
unittests.exercise_3(test_mar_pattern)

<a name='4---missingness-correlations'></a>
## 4 - Missingness Correlations

<a name='41---why-correlations-matter'></a>
### 4.1 - Why Correlations Matter

From the theory — the `missingno` heatmap shows **pairwise nullity correlations**:
- **+1** — the two columns are *always* missing at the same rows (shared upstream cause)
- **-1** — the columns are *never* missing together (mutually exclusive paths)
- **0** — missingness in one column gives no information about the other

This identifies clusters of features that should be imputed jointly (KNN or MICE) rather than column-wise.

<a name='exercise-4---compute_missingness_correlation'></a>
### **Exercise 4 - `compute_missingness_correlation`**

**Your Task:**

Implement `compute_missingness_correlation(df)` that computes the correlation matrix of the missingness indicators for all columns that have at least one missing value.

**Requirements:**
- Only include columns that have at least one missing value
- Create binary indicators and compute their Pearson correlation matrix
- Return a `pd.DataFrame` (correlation matrix, values in −1 to +1)

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** `cols_with_missing = df.columns[df.isnull().any()].tolist()`

**Step 2:** Create indicators for those columns: `df[cols_with_missing].isnull().astype(int)`

**Step 3:** `.corr()` on the indicator DataFrame returns the correlation matrix

</details>

In [ ]:
# GRADED FUNCTION: compute_missingness_correlation
def compute_missingness_correlation(df):
    """
    Compute pairwise correlations between missingness indicators.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        pd.DataFrame: Correlation matrix of missingness indicators.
    """
    ### START CODE HERE ###

    # Step 1: Identify columns with at least one missing value
    cols_with_missing = None

    # Step 2: Create binary indicators
    indicators = None

    # Step 3: Compute correlation matrix
    corr_matrix = None

    ### END CODE HERE ###

    return corr_matrix

In [ ]:
# Check on MCAR data — low correlations expected (random missingness)
corr_mcar = compute_missingness_correlation(mcar_data)
print("Missingness correlation matrix (MCAR data):")
print(corr_mcar.round(3))

In [ ]:
# Test your code!
unittests.exercise_4(compute_missingness_correlation)

<a name='5---classifying-missingness-type'></a>
## 5 - Classifying Missingness Type

Combining the tests above, we can build a heuristic classifier that labels a column's missingness as likely MCAR, MAR, or MNAR. The classification logic follows the theory's decision framework:

1. Run simplified MCAR test across predictor columns
2. If no predictor produces a significant p-value → likely **MCAR**
3. If at least one predictor shows strong correlation → likely **MAR**
4. If the column's own lagged/related values suggest self-dependency → flag as potential **MNAR**

<a name='exercise-5---classify_missingness_type'></a>
### **Exercise 5 - `classify_missingness_type`**

**Your Task:**

Implement `classify_missingness_type(df, target_col, significance_level=0.05)` that returns a string classification: `'MCAR'`, `'MAR'`, or `'MNAR'`.

**Requirements:**
- Test each numeric column in `df` (excluding `target_col`) as a potential predictor
- Use `simplified_mcar_test` for group mean differences
- Use `test_mar_pattern` for correlation-based MAR signal
- Return `'MCAR'` if no predictor shows significance; `'MAR'` if at least one predictor correlates; `'MNAR'` if evidence is ambiguous / high self-correlation

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Step 1:** Get numeric columns excluding `target_col`

**Step 2:** For each predictor, run both `simplified_mcar_test` and `test_mar_pattern`

**Step 3:** Count significant results. If any correlation p-value < threshold → `'MAR'`. If no p-value is significant → `'MCAR'`. Otherwise → `'MNAR'`.

</details>

In [ ]:
# GRADED FUNCTION: classify_missingness_type
def classify_missingness_type(df, target_col, significance_level=0.05):
    """
    Classify the likely missingness mechanism for a column.

    Args:
        df (pd.DataFrame): Input DataFrame.
        target_col (str): Column to classify.
        significance_level (float): P-value threshold. Default 0.05.

    Returns:
        str: One of 'MCAR', 'MAR', or 'MNAR'.
    """
    ### START CODE HERE ###

    # Step 1: Get numeric predictor columns (excluding target)
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    predictors = [c for c in numeric_cols if c != target_col]

    mar_signals = 0
    mcar_consistent = 0

    for predictor in predictors:
        # Skip if predictor has no variance after dropping NaN
        if df[predictor].dropna().std() == 0:
            continue

        # Step 2a: Group mean difference test
        try:
            _, p_ttest = simplified_mcar_test(df, target_col, predictor)
        except Exception:
            continue

        # Step 2b: Correlation test
        try:
            _, p_corr = test_mar_pattern(df, target_col, predictor)
        except Exception:
            continue

        if p_ttest < significance_level or p_corr < significance_level:
            mar_signals += 1
        else:
            mcar_consistent += 1

    # Step 3: Classification rule
    if mar_signals == 0:
        classification = None  # No predictor triggers significance -> MCAR
    elif mar_signals > 0 and mar_signals <= len(predictors) // 2:
        classification = None  # Some predictors significant -> MAR
    else:
        classification = None  # Widespread significance or ambiguous -> MNAR

    ### END CODE HERE ###

    return classification

In [ ]:
print("MCAR data classification:", classify_missingness_type(mcar_data, 'income'))
print("MAR data  classification:", classify_missingness_type(mar_data,  'income'))
print("MNAR data classification:", classify_missingness_type(mnar_data, 'income'))

In [ ]:
# Test your code!
unittests.exercise_5(classify_missingness_type)